#Proyecto: Agente Inteligente para Clínica Salud

##Objetivo final

Construir un asistente inteligente basado en IA capaz de responder consultas de pacientes utilizando exclusivamente la documentación oficial de la clínica mediante una arquitectura RAG (Retrieval-Augmented Generation).

# 1.- Diseño y Planificación del Proyecto

##Este proyecto propone el desarrollo de un Agente Inteligente basado en Inteligencia Artificial Generativa y Recuperación Aumentada por Recuperación (RAG), capaz de responder consultas utilizando exclusivamente la documentación oficial de la clínica, garantizando respuestas coherentes, consistentes y alineadas con la información institucional.

##El agente actuará como un asistente virtual disponible las 24 horas del día para orientar a pacientes y usuarios, sin reemplazar el criterio médico ni emitir diagnósticos.

# Problema

### Actualmente, los pacientes enfrentan diversas dificultades al intentar obtener información sobre los servicios de una clínica, entre ellas:

### Saturación de líneas telefónicas.


*   Largos tiempos de espera.
*   Información distribuida en múltiples documentos.
*   Respuestas inconsistentes entre distintos canales de atención.
*   Dependencia del horario de funcionamiento del personal administrativo.


### Estas situaciones afectan la experiencia del paciente y aumentan la carga de trabajo del personal de recepción.

In [1]:
# ==========================================
# INSTALACIÓN LIBRERÍAS RAG
# ==========================================


!pip install -q \
langchain \
langchain-community \
langchain-classic \
langchain-text-splitters \
pypdf \
faiss-cpu \
cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import pypdf
import faiss
import sentence_transformers
import cohere

print("Entorno RAG listo correctamente")

Entorno RAG listo correctamente


In [4]:
# ==========================================
# CREAR ESTRUCTURA PROYECTO LOCAL COLAB
# ==========================================

from pathlib import Path


carpetas = [

    "documentos_clinica",
    "dataset",
    "vector_db",
    "resultados"

]


for carpeta in carpetas:

    Path(carpeta).mkdir(
        exist_ok=True
    )

    print(
        "✅ Creada:",
        carpeta
    )


print("\nEstructura lista para pruebas")

✅ Creada: documentos_clinica
✅ Creada: dataset
✅ Creada: vector_db
✅ Creada: resultados

Estructura lista para pruebas


In [5]:
# ==========================================
# LECTURA DOCUMENTOS PDF
# ==========================================

from pathlib import Path
from pypdf import PdfReader


documentos_cargados = []


for archivo in Path(
    "documentos_clinica"
).glob("*.pdf"):


    lector = PdfReader(
        archivo
    )


    texto = ""


    for pagina in lector.pages:

        texto += pagina.extract_text()


    documentos_cargados.append({

        "contenido": texto,

        "fuente": archivo.name

    })


print(
    "Documentos cargados:",
    len(documentos_cargados)
)

PDF encontrados: 5
- 01_Politica_Privacidad_Datos_Paciente.pdf
- 04_Guia_Convenios_Coberturas.pdf
- 02_FAQ_Consultas_Turnos.pdf
- 05_Instrucciones_Pre_Post_Consulta.pdf
- 03_Politica_Cancelaciones_Reagendamiento.pdf


In [8]:
# ==========================================
# CONFIGURACIÓN GENERAL DEL PROYECTO
# ==========================================

CONFIG = {

    "proyecto":
    "Agente IA Clínica Horizonte",

    "version":
    "1.0",

    "tipo":
    "RAG Atención Paciente",

    "fuente_documentos":
    "PDF Clínica",

    "cantidad_documentos":
    5,

    "vector_database":
    "FAISS",

    "embedding_model":
    "sentence-transformers/all-MiniLM-L6-v2",

    "chunk_size":
    1000,

    "chunk_overlap":
    100,

    "dataset_pruebas":
    "Dataset_pruebas.csv"

}


CONFIG

{'proyecto': 'Agente IA Clínica Horizonte',
 'version': '1.0',
 'tipo': 'RAG Atención Paciente',
 'fuente_documentos': 'PDF Clínica',
 'cantidad_documentos': 5,
 'vector_database': 'FAISS',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'chunk_size': 1000,
 'chunk_overlap': 100,
 'dataset_pruebas': 'Dataset_pruebas.csv'}

##División de texto y preparación del conocimiento

In [9]:
# ==========================================
# IMPORTAR HERRAMIENTAS DE PROCESAMIENTO
# ==========================================

from langchain_text_splitters import RecursiveCharacterTextSplitter


print(
    "✅ Herramientas cargadas"
)

✅ Herramientas cargadas


In [10]:
# ==========================================
# CARGAR DOCUMENTOS PDF
# ==========================================

from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path


documentos_clinica = []


for archivo in Path(
    "documentos_clinica"
).glob("*.pdf"):

    loader = PyPDFLoader(
        str(archivo)
    )

    documentos = loader.load()

    documentos_clinica.extend(
        documentos
    )


print(
    "Páginas cargadas:",
    len(documentos_clinica)
)

/tmp/ipykernel_536/3392325654.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Páginas cargadas: 12


In [11]:
# ==========================================
# CONFIGURACIÓN CHUNKS
# ==========================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=CONFIG["chunk_size"],

    chunk_overlap=CONFIG["chunk_overlap"],

    length_function=len

)


print(
    "✅ Fragmentador configurado"
)

✅ Fragmentador configurado


In [12]:
# ==========================================
# CREAR FRAGMENTOS
# ==========================================

fragmentos = splitter.split_documents(

    documentos_clinica

)


print(
    "Cantidad de fragmentos:",
    len(fragmentos)
)

Cantidad de fragmentos: 22


In [13]:
# ==========================================
# REVISAR FRAGMENTOS
# ==========================================

for i, fragmento in enumerate(fragmentos):

    print(
        "Fragmento:",
        i+1
    )

    print(
        "Fuente:",
        fragmento.metadata["source"]
    )

    print(
        "Página:",
        fragmento.metadata.get("page")
    )

    print(
        "Tamaño:",
        len(fragmento.page_content),
        "caracteres"
    )

    print("-"*50)

Fragmento: 1
Fuente: documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf
Página: 0
Tamaño: 958 caracteres
--------------------------------------------------
Fragmento: 2
Fuente: documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf
Página: 0
Tamaño: 989 caracteres
--------------------------------------------------
Fragmento: 3
Fuente: documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf
Página: 0
Tamaño: 90 caracteres
--------------------------------------------------
Fragmento: 4
Fuente: documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf
Página: 1
Tamaño: 89 caracteres
--------------------------------------------------
Fragmento: 5
Fuente: documentos_clinica/04_Guia_Convenios_Coberturas.pdf
Página: 0
Tamaño: 947 caracteres
--------------------------------------------------
Fragmento: 6
Fuente: documentos_clinica/04_Guia_Convenios_Coberturas.pdf
Página: 0
Tamaño: 363 caracteres
--------------------------------------------------
Fragmento: 7
Fuente: doc

In [14]:
# ==========================================
# MOSTRAR CONTENIDO Y ORIGEN
# ==========================================

print(
    "Fuente:"
)

print(
    fragmentos[0].metadata["source"]
)


print("\nContenido:")

print(
    fragmentos[0].page_content
)

Fuente:
documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf

Contenido:
Clínica Inteligente Horizonte
Política de Privacidad de Datos del Paciente
Versión 1.0 - Julio 2026
1. Objetivo
Establecer los lineamientos para el tratamiento, almacenamiento y protección de los
datos personales y clínicos de los pacientes, garantizando la confidencialidad,
integridad y disponibilidad de la información.
2. Alcance
Aplica a todo el personal clínico, administrativo, proveedores autorizados y terceros
que, por sus funciones, accedan a información de pacientes.
3. Datos recopilados
La clínica podrá recopilar datos de identificación, contacto, antecedentes médicos,
resultados de exámenes, información de seguros o convenios y registros
administrativos necesarios para la atención.
4. Uso de la información
Los datos serán utilizados exclusivamente para la prestación de servicios de salud,
coordinación de citas, continuidad de tratamientos, procesos administrativos y
cumplimiento de obligaciones l

In [15]:
# ==========================================
# VALIDAR METADATA
# ==========================================


print(
    fragmentos[0].metadata
)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-27T01:50:41+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-27T01:50:41+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [16]:
# ==========================================
# GUARDAR FRAGMENTOS
# ==========================================

import pickle
from pathlib import Path


ruta_fragmentos = Path(
    "resultados/fragmentos_clinica.pkl"
)


with open(
    ruta_fragmentos,
    "wb"
) as archivo:

    pickle.dump(
        fragmentos,
        archivo
    )


print(
    "✅ Fragmentos guardados:",
    ruta_fragmentos
)

✅ Fragmentos guardados: resultados/fragmentos_clinica.pkl


#Creación de Embeddings y Memoria Inteligente FAISS

In [17]:
!pip install -U langchain-huggingface -q

In [18]:
# ==========================================
# INSTALAR HUGGINGFACE EMBEDDINGS
# ==========================================

!pip install -U langchain-huggingface -q

print(
    "✅ Librería embeddings instalada"
)

✅ Librería embeddings instalada


In [19]:
# ==========================================
# IMPORTAR EMBEDDINGS
# ==========================================

from langchain_huggingface import HuggingFaceEmbeddings


print(
    "✅ Librería embeddings cargada"
)

✅ Librería embeddings cargada


In [20]:
# ==========================================
# MODELO EMBEDDING
# ==========================================

modelo_embeddings = HuggingFaceEmbeddings(

    model_name=
    CONFIG["embedding_model"]

)


print(
    "✅ Modelo de embeddings listo"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo de embeddings listo


In [21]:
# ==========================================
# PRUEBA DE VECTOR
# ==========================================

texto_prueba = [

    "¿Cómo puedo cambiar mi hora médica?"

]


vector = modelo_embeddings.embed_documents(

    texto_prueba

)


print(
    "Dimensión del vector:",
    len(vector[0])
)

Dimensión del vector: 384


In [22]:
# ==========================================
# IMPORTAR FAISS
# ==========================================

from langchain_community.vectorstores import FAISS


print(
    "✅ FAISS cargado"
)

✅ FAISS cargado


In [23]:
# ==========================================
# CREAR BASE VECTORIAL FAISS
# ==========================================

base_vectorial = FAISS.from_documents(

    fragmentos,

    modelo_embeddings

)


print(
    "✅ Base vectorial creada correctamente"
)

✅ Base vectorial creada correctamente


In [24]:
# ==========================================
# GUARDAR MEMORIA FAISS
# ==========================================

base_vectorial.save_local(

    "vector_db/base_vectorial_clinica"

)


print(
    "✅ Memoria clínica guardada"
)

✅ Memoria clínica guardada


In [27]:
# ==========================================
# BUSQUEDA SEMANTICA
# ==========================================


consulta = """

¿Cómo puedo cambiar mi hora médica?

"""


resultados = base_vectorial.similarity_search(

    consulta,

    k=3

)


for resultado in resultados:


    print(
        "Fuente:",
        resultado.metadata["source"]
    )


    print(
        resultado.page_content[:300]
    )


    print("-"*50)

Fuente: documentos_clinica/05_Instrucciones_Pre_Post_Consulta.pdf
• Presentar un documento de identidad vigente.  
• Llevar la orden médica, si corresponde.  
• Informar cualquier cambio en sus datos personales.  
• Portar los resultados de exámenes o informes médicos relacionados con la consulta.  
• Informar alergias conocidas y medicamentos que esté utilizando.
--------------------------------------------------
Fuente: documentos_clinica/02_FAQ_Consultas_Turnos.pdf
Clínica Inteligente Horizonte
Preguntas Frecuentes sobre Consultas y Turnos
Versión 1.0 - Julio 2026
¿Cómo solicito una consulta?
Puede solicitar una cita por teléfono, sitio web o de forma presencial.
¿Cómo confirmo mi turno?
Recibirá una confirmación por correo electrónico o teléfono registrado.
¿
--------------------------------------------------
Fuente: documentos_clinica/05_Instrucciones_Pre_Post_Consulta.pdf
• Fiebre alta prolongada.  
• Reacción alérgica a medicamentos.  
• Empeoramiento importante de los síntomas.

In [28]:
# ==========================================
# PRUEBAS MEMORIA SEMÁNTICA
# ==========================================


preguntas = [

"¿Cómo protege la clínica mis datos?",

"¿Qué debo llevar antes de una consulta?",

"¿Puedo cancelar una cita?",

"¿Qué convenios médicos existen?"

]


for pregunta in preguntas:


    print("\nPregunta:")
    print(pregunta)


    respuesta = base_vectorial.similarity_search(

        pregunta,

        k=1

    )


    print(
        "Documento:",
        respuesta[0].metadata["source"]
    )


Pregunta:
¿Cómo protege la clínica mis datos?
Documento: documentos_clinica/01_Politica_Privacidad_Datos_Paciente.pdf

Pregunta:
¿Qué debo llevar antes de una consulta?
Documento: documentos_clinica/02_FAQ_Consultas_Turnos.pdf

Pregunta:
¿Puedo cancelar una cita?
Documento: documentos_clinica/03_Politica_Cancelaciones_Reagendamiento.pdf

Pregunta:
¿Qué convenios médicos existen?
Documento: documentos_clinica/05_Instrucciones_Pre_Post_Consulta.pdf


#Creación del Retriever y Motor de Búsqueda del Paciente

In [29]:
# ==========================================
# CREAR RETRIEVER CLÍNICO
# ==========================================


retriever = base_vectorial.as_retriever(

    search_kwargs={

        "k": 3

    }

)


print(
    "✅ Retriever creado correctamente"
)

✅ Retriever creado correctamente


In [30]:
# ==========================================
# PRUEBA RETRIEVER
# ==========================================


pregunta = """

¿Qué debo hacer antes de mi consulta?

"""


documentos_encontrados = retriever.invoke(

    pregunta

)


print(
    "Documentos encontrados:",
    len(documentos_encontrados)
)

Documentos encontrados: 3


In [31]:
# ==========================================
# MOSTRAR RESULTADOS RETRIEVER
# ==========================================


for i, documento in enumerate(documentos_encontrados):


    print(
        "Resultado:",
        i+1
    )


    print(
        "Fuente:",
        documento.metadata["source"]
    )


    print(
        documento.page_content[:400]
    )


    print("-"*60)

Resultado: 1
Fuente: documentos_clinica/02_FAQ_Consultas_Turnos.pdf
Clínica Inteligente Horizonte
Preguntas Frecuentes sobre Consultas y Turnos
Versión 1.0 - Julio 2026
¿Cómo solicito una consulta?
Puede solicitar una cita por teléfono, sitio web o de forma presencial.
¿Cómo confirmo mi turno?
Recibirá una confirmación por correo electrónico o teléfono registrado.
¿Puedo cambiar la fecha?
Sí, sujeto a disponibilidad y a la política de reagendamiento.
¿Qué pasa si 
------------------------------------------------------------
Resultado: 2
Fuente: documentos_clinica/05_Instrucciones_Pre_Post_Consulta.pdf
Dependiendo del motivo de la consulta, el profesional tratante podrá indicar medidas 
adicionales, tales como: 
• Ayuno previo para determinados exámenes.  
• Suspensión temporal de algunos medicamentos.  
• Uso de ropa cómoda para facilitar la evaluación médica.  
• Llevar dispositivos médicos personales, como lentes, audífonos o equipos de monitoreo.  
Estas indicaciones serán comunicad

In [32]:
# ==========================================
# FUNCIÓN BUSCADOR CLÍNICO
# ==========================================


def buscar_documentos(pregunta):


    resultados = retriever.invoke(

        pregunta

    )


    return resultados



print(
    "✅ Función creada"
)

✅ Función creada


In [33]:
# ==========================================
# INSTALAR COHERE
# ==========================================

!pip install -q langchain-cohere cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 10.0 MB/s eta 0:00:00


In [ ]:
# ==========================================
# CONFIGURAR API KEY
# ==========================================


import os

from getpass import getpass


os.environ["COHERE_API_KEY"] = getpass(

    "Ingrese API Key Cohere: "

)


print(
    "✅ API configurada"
)

In [ ]:
# ==========================================
# CREAR MODELO IA
# ==========================================


from langchain_cohere import ChatCohere


llm = ChatCohere(

    model="command-a-03-2025",

    temperature=0

)


print(
    "✅ Modelo IA conectado"
)

In [ ]:
# ==========================================
# PROMPT DEL AGENTE
# ==========================================


from langchain_core.prompts import PromptTemplate



prompt_clinica = PromptTemplate(


    input_variables=[

        "context",

        "question"

    ],


    template="""


Eres el asistente virtual
de Clínica Inteligente Horizonte.


Tu función es ayudar a pacientes
con información administrativa.


Reglas:

- Usa únicamente el contexto entregado.
- No inventes información.
- Si no encuentras la respuesta indica:

"No encuentro esa información
en los documentos disponibles."


Contexto:

{context}


Pregunta del paciente:

{question}


Respuesta:


"""

)


print(
    "✅ Prompt clínico creado"
)

In [ ]:
# ==========================================
# CREAR AGENTE RAG
# ==========================================


from langchain_classic.chains import RetrievalQA



agente_clinico = RetrievalQA.from_chain_type(


    llm=llm,


    retriever=retriever,


    return_source_documents=True,


    chain_type_kwargs={

        "prompt":

        prompt_clinica

    }


)



print(
    "✅ Agente RAG creado"
)

In [ ]:
# ==========================================
# PRUEBA AGENTE
# ==========================================


pregunta = """

¿Cómo puedo cambiar mi hora médica?

"""


respuesta = agente_clinico.invoke(

    {

        "query":

        pregunta

    }

)


print(

    respuesta["result"]

)

In [ ]:
# ==========================================
# FUENTES UTILIZADAS
# ==========================================


for documento in respuesta["source_documents"]:


    print(

        documento.metadata["source"]

    )

In [ ]:
# ==========================================
# PRUEBAS AGENTE
# ==========================================


preguntas = [

"¿Cómo protege la clínica mis datos?",

"¿Qué debo llevar a mi consulta?",

"¿Puedo cambiar mi hora médica?",

"¿Qué seguros acepta la clínica?"

]


for pregunta in preguntas:


    resultado = agente_clinico.invoke(

        {

            "query":

            pregunta

        }

    )


    print("="*70)

    print("Paciente:")

    print(pregunta)


    print("\nAsistente:")

    print(

        resultado["result"]

    )